# Wave Equation and C Code Generation

Turn the scalar-wave right-hand sides into NRPy symbolic expressions and complete C assignments.

Navigation: [Index](../index.ipynb) | Previous: [Finite-Difference Playground](../2-numerical_methods/finite_difference_playground.ipynb) | Next: [Start-to-Finish Cartesian Wave Project](start_to_finish_wave_cartesian.ipynb)

## Roadmap

This notebook proceeds through short focused cells, each with one action and one interpretable output.

## Symbolic Wave Right-Hand Sides
The first-order scalar wave system uses `uu_t = vv` and `vv_t = wavespeed**2` times the spatial Laplacian of `uu`.

## Step 1: Import modules

These imports expose the NRPy and Python tools used in the next steps.

In [1]:
import sympy as sp

## Step 2: Import NRPy modules

These imports expose the NRPy registries and infrastructure writers used below.

In [2]:
import nrpy.c_codegen as ccg
import nrpy.grid as grid
import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.wave_equation.WaveEquation_RHSs import WaveEquation_RHSs
from nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData import WaveEquation_solution_Cartesian

## Step 3: Build the Cartesian wave-equation RHSs

The printed right-hand sides are the symbolic expressions later passed to C code generation.

In [3]:
for name in ["uu", "vv", "uu_rhs", "vv_rhs"]:
    grid.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "BHaH")
rhs = WaveEquation_RHSs()
print("uu_rhs =", rhs.uu_rhs)
print("vv_rhs =", rhs.vv_rhs)

uu_rhs = vv
vv_rhs = uu_dDD00*wavespeed**2 + uu_dDD11*wavespeed**2 + uu_dDD22*wavespeed**2


## Step 4: Check the residual

A zero residual confirms that the symbolic expression matches the expected identity.

In [4]:
wavespeed = sp.Symbol("wavespeed", real=True)
vv = sp.Symbol("vv", real=True)
uu_dDD = ixp.declarerank2("uu_dDD", symmetry="sym01")
hand_uu_rhs = vv
hand_vv_rhs = wavespeed**2 * sum(uu_dDD[i][i] for i in range(3))
print("uu residual:", sp.simplify(rhs.uu_rhs - hand_uu_rhs))
print("vv residual:", sp.simplify(rhs.vv_rhs - hand_vv_rhs))
if sp.simplify(rhs.uu_rhs - hand_uu_rhs) != 0 or sp.simplify(rhs.vv_rhs - hand_vv_rhs) != 0:
    raise RuntimeError("Expected the NRPy RHSs to match the hand-written RHSs.")

uu residual: 0
vv residual: 0


## Step 5: Generate C code

The output shows how a symbolic expression is emitted as C.

In [5]:
print("complete RHS assignments:")
print(ccg.c_codegen([rhs.uu_rhs, rhs.vv_rhs], ["uu_rhs", "vv_rhs"], include_braces=False, verbose=False))

complete RHS assignments:
const REAL tmp0 = ((wavespeed)*(wavespeed));
uu_rhs = vv;
vv_rhs = tmp0*uu_dDD00 + tmp0*uu_dDD11 + tmp0*uu_dDD22;



## Step 6: Generate C code

The output shows how a symbolic expression is emitted as C.

In [6]:
plane_wave = WaveEquation_solution_Cartesian(WaveType="PlaneWave")
print("complete exact-solution assignments:")
print(ccg.c_codegen([plane_wave.uu_exactsoln, plane_wave.vv_exactsoln], ["uu_exact", "vv_exact"], include_braces=False, verbose=False))

complete exact-solution assignments:
const REAL tmp0 = time*wavespeed - (kk0*xCart0 + kk1*xCart1 + kk2*xCart2)/sqrt(((kk0)*(kk0)) + ((kk1)*(kk1)) + ((kk2)*(kk2)));
uu_exact = 2 - sin(tmp0);
vv_exact = -wavespeed*cos(tmp0);



## Interpreting the Output
The residuals confirm that the generated symbolic right-hand sides match the hand-written wave system. The C assignments are the compact kernel body used by complete projects.

## Where This Leads
- [C Code Generation](../1-intro/c_codegen.ipynb)
- [Finite Differences](../1-intro/finite_difference.ipynb)
- [Start-to-Finish Cartesian Wave Project](start_to_finish_wave_cartesian.ipynb)